In [ ]:
## Load Environment Variables
import os
from dotenv import load_dotenv
load_dotenv()
print("Environment variables loaded")

In [ ]:
## Imports from Langchain
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from typing import List

In [ ]:
## Custom Pre-Processor Function to Process the PDF
class SmartPDFProcessor:
    """Advanced PDF processing with error handling"""

    def __init__(self, chunk_size=1000, chunk_overlap=100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=[" "],
        )

    def process_pdf(self, pdf_path: str) -> List[Document]:
        """Process PDF with smart chunking and metadata enhancement"""

        # Load PDF
        loader = PyMuPDFLoader(pdf_path)
        pages = loader.load()

        # Process each page
        processed_chunks = []

        for page_num, page in enumerate(pages):
            # Clean text
            cleaned_text = self._clean_text(page.page_content)

            # Skip nearly empty pages
            if len(cleaned_text.strip()) < 50:
                continue

            # Create chunks with enhanced metadata
            chunks = self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[{
                    **page.metadata,
                    "page": page_num + 1,
                    "total_pages": len(pages),
                    "chunk_method": "smart_pdf_processor",
                    "char_count": len(cleaned_text),
                }],
            )

            processed_chunks.extend(chunks)

        return processed_chunks

    def _clean_text(self, text: str) -> str:
        """Clean extracted text"""
        # Remove excessive whitespace
        text = " ".join(text.split())

        # Fix common PDF extraction issues
        text = text.replace("\ufb01", "fi")
        text = text.replace("\ufb02", "fl")

        return text

In [ ]:
preprocessor=SmartPDFProcessor()

In [ ]:
preprocessor

In [ ]:
## Process a PDF with Pre-Processor Function and Create Chunks
try:
    smart_chunks = preprocessor.process_pdf("Data/annual-report.pdf")
    print(f"Processed into {len(smart_chunks)} smart chunks")

    # Show enhanced metadata
    if smart_chunks:
        print("\nSample chunk metadata:")
        for key, value in smart_chunks[0].metadata.items():
            print(f"  {key}: {value}")

except Exception as e:
    print(f"Processing error: {e}")

In [ ]:
# Print the chunks
print(smart_chunks[0])
print("-----------------------------")
print(smart_chunks[5])
print("-----------------------------")
print(smart_chunks[11])
print("-----------------------------")
print(smart_chunks[41])

In [ ]:
## Import required modules (LATEST imports)
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

In [ ]:
## Initialize HuggingFace Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},  # change to "cuda" if GPU available
    encode_kwargs={"normalize_embeddings": True},
)

In [ ]:
## Create FAISS Vector Store from smart_chunks
vectorstore = FAISS.from_documents(
    documents=smart_chunks,
    embedding=embeddings
)
print(f"\n Embeddings are created and stored in memory.")

In [ ]:
## Save FAISS Index to Local Disk

FAISS_INDEX_PATH = "faiss_index"

vectorstore.save_local(FAISS_INDEX_PATH)

print("Vector store saved to 'faiss_index' directory")

In [ ]:
## Load FAISS Index Later

loaded_vectorstore = FAISS.load_local(
    FAISS_INDEX_PATH,
    embeddings,
    allow_dangerous_deserialization=True
)

print(f"Loaded vector store contains {loaded_vectorstore.index.ntotal} vectors")

In [ ]:
## Quick Similarity Test
query = "What is this document about?"
docs = loaded_vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(docs, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content[:300])

In [ ]:
load_dotenv()

In [ ]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")    # Load API Key

In [ ]:
## Define GROQ LLM
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="groq:openai/gpt-oss-120b"
)

In [ ]:
llm.invoke("Explain what is LangChain in one paragraph.")  # Test Query to LLM

In [ ]:
# Define Prompt Template
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context: {context}

Question: {question}

Answer:""")

In [ ]:
# Define Retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [ ]:
from typing import List
# Format documents for the prompt
def format_docs(docs: List[Document]) -> str:
    """Format documents for insertion into prompt"""
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'Unknown')
        formatted.append(f"Document {i+1} (Source: {source}):\n{doc.page_content}")
    return "\n\n".join(formatted)

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

## Create RAG Chain

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | simple_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
rag_chain

In [ ]:
response = rag_chain.invoke(
    "What were DBS's net profit and total assets?"
)
print(response)

In [ ]:
response = rag_chain.invoke(
    "How much sustainable financing has DBS committed to, and how many individuals did DBS Foundation's programmes uplift?"
)
print(response)

In [ ]:
response = rag_chain.invoke(
    "What is the capital of France?"
)
print(response)

In [ ]:
## Hybrid prompt: document first, general knowledge second
hybrid_prompt = ChatPromptTemplate.from_template("""You are a helpful assistant. You are given context retrieved from the user's document.

Rules:
1. If the context contains the answer, use it first and say it comes from the document.
2. If the context only partly answers the question, give the part from the document, then add what you know from general knowledge, clearly labeled "General knowledge".
3. If the context has nothing relevant, answer from general knowledge and say the document doesn't cover it.
4. Never present general knowledge as if it came from the document. If you are unsure, say so.
5. Start with a one-sentence direct answer, then give details.
6. Keep the answer short. Use a table only when comparing several items.
7. Copy numbers and currencies exactly as written in the document. Never convert or change the currency.

Context:
{context}

Question: {question}

Answer:""")

hybrid_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | hybrid_prompt
    | llm
    | StrOutputParser()
)

In [ ]:
print(hybrid_chain.invoke("What is the capital of France?"))
print("-----------------------------")
print(hybrid_chain.invoke("What were DBS's net profit and total assets, and how does that compare to a typical large bank?"))

In [ ]:
from IPython.display import Markdown, display

def ask(chain, question):
    answer = chain.invoke(question)
    display(Markdown(answer))

ask(hybrid_chain, "What were DBS's net profit and total assets, and how does that compare to a typical large bank?")

In [ ]:
docs = retriever.invoke("What was DBS's net profit?")
for d in docs:
    print(f"--- Page {d.metadata.get('page')} ---")
    print(d.page_content[:600], "\n")

In [ ]:
docs = retriever.invoke("What was DBS's net profit?")
for d in docs:
    print(f"--- Page {d.metadata.get('page')} ---")
    print(d.page_content[:1000], "\n")